# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
import sys
!{sys.executable} -m pip install langchain langchain-community langchain-experimental langchain-openai langchain-classic

ERROR: Could not find a version that satisfies the requirement langchain.docstore (from versions: none)
ERROR: No matching distribution found for langchain.docstore

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip


In [1]:
from typing import Optional

# Legacy chains/prompts → langchain-classic
from langchain_classic.chains import LLMChain
from langchain_classic.prompts import PromptTemplate

# BabyAGI still in langchain-experimental
from langchain_experimental.autonomous_agents import BabyAGI

# OpenAI integrations
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [8]:
import sys
!{sys.executable} -m 
%pip install faiss-cpu > /dev/null
%pip install google-search-results > /dev/null


Argument expected for the -m option
usage: /Library/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip available: 22.3 -> 26.1.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [6]:
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

ModuleNotFoundError: No module named 'langchain.docstore'

In [8]:
import sys
!{sys.executable} -m
!pip install faiss-cpu langchain-community

Argument expected for the -m option
usage: /Library/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.


In [9]:
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore  # Updated path
from langchain_community.vectorstores import FAISS

In [10]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')
SERPAPI_API_KEY = os.getenv('SERPAPI_API_KEY')

In [11]:
# Define your embedding model
from langchain_openai import OpenAIEmbeddings
embeddings_model = OpenAIEmbeddings()

# Initialize the vectorstore as empty
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)
vectorstore = FAISS(
    embedding_function=embeddings_model,  # Full embeddings object, not .embed_query
    index=index,
    docstore=InMemoryDocstore({}),
    index_to_docstore_id={}
)

## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [12]:
import sys
!{sys.executable} -m
!pip install google-search-results

Argument expected for the -m option
usage: /Library/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python [option] ... [-c cmd | -m mod | file | -] [arg] ...
Try `python -h' for more information.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for google-search-results: filename=google_search_results-2.4.2-py3-none-any.whl size=32093 sha256=fd1d54aea0745919956473e6a17f4c5827a8c070dfc8f6c4aab6062b1d6c4001
  Stored in directory: /Users/sazao/Library/Caches/pip/wheels/44/af/e2/dde9fab6f1876485b72b35e9cd48da741da67d20e617c3b971
Successfully built google-search-results


In [13]:
from langchain_classic.agents import AgentExecutor, Tool, ZeroShotAgent  # Legacy agents
from langchain_classic.chains import LLMChain  # Legacy chains
from langchain_classic.prompts import PromptTemplate  # Legacy prompts
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI
from langchain_openai import ChatOpenAI

todo_prompt = PromptTemplate.from_template(
    "You are a planner who is an expert at coming up with a todo list for a given objective. Come up with a todo list for this objective: {objective}"
)
todo_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), prompt=todo_prompt)
search = SerpAPIWrapper()
tools = [
    Tool(
        name="Search",
        func=search.run,
        description="useful for when you need to answer questions about current events",
    ),
    Tool(
        name="TODO",
        func=todo_chain.run,
        description="useful for when you need to come up with todo lists. Input: an objective to create a todo list for. Output: a todo list for that objective. Please be very clear what the objective is!",
    ),
]


prefix = """You are an AI who performs one task based on the following objective: {objective}. Take into account these previously completed tasks: {context}."""
suffix = """Question: {task}
{agent_scratchpad}"""
prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)

/var/folders/rn/hjmgpq6j4bl5hk8kkpw6k1f40000gn/T/ipykernel_12815/349246153.py:11: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  todo_chain = LLMChain(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0), prompt=todo_prompt)


In [14]:

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_chain = LLMChain(llm=llm, prompt=prompt)
tool_names = [tool.name for tool in tools]
agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent, tools=tools, verbose=True
)

/var/folders/rn/hjmgpq6j4bl5hk8kkpw6k1f40000gn/T/ipykernel_12815/1591959347.py:4: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent = ZeroShotAgent(llm_chain=llm_chain, allowed_tools=tool_names)


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [15]:
OBJECTIVE = "Write a weather report for SF today"

In [16]:
# Logging of LLMChains
verbose = False
# If None, will keep on going forever
max_iterations: Optional[int] = 3
baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)

In [17]:
baby_agi({"objective": OBJECTIVE})

/var/folders/rn/hjmgpq6j4bl5hk8kkpw6k1f40000gn/T/ipykernel_12815/3867971396.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  baby_agi({"objective": OBJECTIVE})



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Question: What is the weather report for San Francisco today?
Thought: I need to find the current weather conditions for San Francisco to provide an accurate report.
Action: Search
Action Input: "current weather in San Francisco today"
Observation: {'type': 'weather_result', 'temperature': '63', 'unit': 'Fahrenheit', 'precipitation': '5%', 'humidity': '75%', 'wind': '18 mph', 'location': 'San Francisco, CA', 'date': 'Wednesday', 'weather': 'Sunny'}
Thought:I now have the current weather conditions for San Francisco, which will allow me to create a detailed weather report.

Final Answer: Today in San Francisco, the weather is sunny with a temperature of 63°F. The humidity is at 75%, and there is a light wind blowing at 18 mph. There is a low chance of precipitation at 5%. Enjoy the beautiful day!

> Finished chain.

*****TASK RESULT*****

Today in San Francisco, th

{'objective': 'Write a weather report for SF today'}